In [ ]:
#Importacion de librerias necesarias para el proyecto
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Librerías para visualización
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:

#  Cargar JSON
df = pd.read_json('TelecomX_Data.json')
#df.head()

# Expandir cada columna anidada
customer_expanded = pd.json_normalize(df['customer'])
phone_expanded = pd.json_normalize(df['phone'])
internet_expanded = pd.json_normalize(df['internet'])
account_expanded = pd.json_normalize(df['account'])

# Combinar todo 
df_final = pd.concat([
    df[['customerID', 'Churn']],
    customer_expanded,
    phone_expanded,
    internet_expanded,
    account_expanded
], axis=1)

df_final.head()

In [ ]:
# Verificar si 'charges' está anidado y expandirlo
if 'charges' in df_final.columns:
    charges_expanded = pd.json_normalize(df_final['charges'])
    df_final = pd.concat([df_final.drop('charges', axis=1), charges_expanded], axis=1)
  
df = df_final  

In [ ]:
#  ELIMINAR CUSTOMERID (irrelevante)
df = df.drop('customerID', axis=1)


# LIMPIAR CHURN (eliminar filas con valor vacío)
filas_iniciales = len(df)
df = df[df['Churn'] != ''].copy()

#  LIMPIAR Y RELLENAR CHARGES.TOTAL
# Convertir a numérico (los espacios ' ' se vuelven NaN)
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
nan_count = df['Charges.Total'].isna().sum()
# Rellenar NaN con 0 (clientes nuevos)
df['Charges.Total'] = df['Charges.Total'].fillna(0)


#  TRANSFORMAR VARIABLES BINARIAS (Yes/No → 1/0)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


#  TRANSFORMAR GENDER (Female/Male → 1/0)
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})


#  TRANSFORMAR MULTIPLELINES (One-Hot Encoding)
multiple_dummies = pd.get_dummies(df['MultipleLines'], prefix='MultipleLines', drop_first=True)
df = pd.concat([df, multiple_dummies], axis=1)
df = df.drop('MultipleLines', axis=1)


# TRANSFORMAR INTERNETSERVICE (One-Hot Encoding)
internet_dummies = pd.get_dummies(df['InternetService'], prefix='InternetService', drop_first=True)
df = pd.concat([df, internet_dummies], axis=1)
df = df.drop('InternetService', axis=1)

# TRANSFORMAR SERVICIOS DE INTERNET
internet_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                     'TechSupport', 'StreamingTV', 'StreamingMovies']

for service in internet_services:
    dummies = pd.get_dummies(df[service], prefix=service, drop_first=True)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(service, axis=1)


# TRANSFORMAR CONTRACT (One-Hot Encoding)
contract_dummies = pd.get_dummies(df['Contract'], prefix='Contract', drop_first=True)
df = pd.concat([df, contract_dummies], axis=1)
df = df.drop('Contract', axis=1)


# TRANSFORMAR PAYMENTMETHOD (One-Hot Encoding)
payment_dummies = pd.get_dummies(df['PaymentMethod'], prefix='PaymentMethod', drop_first=True)
df = pd.concat([df, payment_dummies], axis=1)
df = df.drop('PaymentMethod', axis=1)


# TRANSFORMAR CHURN (variable objetivo)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
